In [13]:
# Document chunking
# Persistent vector DB (FAISS saved to disk)
# LangChain retriever
# LCEL pipeline (|)
# Gemini 2.5 Flash answering

In [1]:
!pip install langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers

In [2]:
import os
import google.generativeai as genai

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
# -----------------------------
# 1. Configure Gemini API
# -----------------------------

#os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
#genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [5]:
# -----------------------------
# 2. Sample Documents
# -----------------------------

documents = [
    Document(page_content="""Tesla produces electric cars and energy storage products.
    The company was founded by Elon Musk and focuses on sustainable transportation."""),

    Document(page_content="""Toyota is a Japanese automobile manufacturer known for
    hybrid vehicles such as the Prius. The company pioneered hybrid technology."""),

    Document(page_content="""BMW is a German manufacturer of luxury vehicles and
    motorcycles. It is known for performance and premium engineering.""")
]


In [6]:
# -----------------------------
# 3. Document Chunking
# -----------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [7]:
# -----------------------------
# 4. Embedding Model
# -----------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipykernel_1044/3475878687.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:


# -----------------------------
# 5. Persistent Vector Database
# -----------------------------

db_path = "vector_db"

if os.path.exists(db_path):

    vector_db = FAISS.load_local(
        db_path,
        embedding_model,
        allow_dangerous_deserialization=True
    )

else:

    vector_db = FAISS.from_documents(docs, embedding_model)

    vector_db.save_local(db_path)



In [9]:

# -----------------------------
# 6. Create Retriever
# -----------------------------

retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)


# -----------------------------
# 7. Create Gemini LLM
# -----------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [11]:
# -----------------------------
# 8. Prompt Template
# -----------------------------

prompt = ChatPromptTemplate.from_template(
"""
You are an AI assistant answering questions using retrieved context.

Context:
{context}

Question:
{question}

Answer the question clearly using only the provided context.
"""
)


# -----------------------------
# 9. Helper Function
# -----------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# -----------------------------
# 10. LCEL RAG Pipeline
# -----------------------------

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [12]:

# -----------------------------
# 11. Interactive Query Loop
# -----------------------------

while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(question)

    print("\nAnswer:\n")
    print(response.content)


Ask a question (type 'exit' to stop): does bmw manufacture racing cars?

Answer:

The provided context does not state whether BMW manufactures racing cars. It mentions that BMW is a manufacturer of luxury vehicles and motorcycles, known for performance and premium engineering.

Ask a question (type 'exit' to stop): who is BMW?

Answer:

BMW is a German manufacturer of luxury vehicles and motorcycles. It is known for performance and premium engineering.

Ask a question (type 'exit' to stop): exit
